# here is a set of functions help the modification of the database a little bit easier.
Cinyu Zhu, Hopkins, June 2025

In [3]:
import numpy as np

In [3]:
def generate_sql_block():
    temperatures =  ["High", "Low"] 
    table_name = "MODULE_UNDERGROUND2"
    amps = ["A", "B", "C", "D"]

    # image_number = 1
    # suffixes =  ["Tracks", "Defects", "Noise", "Comments", "Reference"]
    # types =     ["VARCHAR(4)", "VARCHAR(4)", "FLOAT", "TEXT", "VARCHAR(100)"]

    # image_number = 2
    # suffixes =  ["Column_Defects", "Noise", "Comments", "Reference"]
    # types =     ["INT", "FLOAT", "TEXT", "VARCHAR(100)"]

    image_number = 32 # 31
    suffixes =  ["Res", "Gain", "Dark_Current"]
    types =     ["FLOAT", "FLOAT", "FLOAT"]

    # image_number = 4
    # suffixes =  ["Pixel_Defects", "Column_Defects", "Region_Defect", "Noise", "CTI_Code", "CTI_Visual", "Sharpness_Tracks", "Comments", "Reference"]
    # types =     ["INT", "INT", "VARCHAR(40)", "FLOAT", "VARCHAR(40)", "VARCHAR(4)", "VARCHAR(4)",  "TEXT", "VARCHAR(100)"]

    # image_number = 4
    # suffixes = ["Defects", "CTI_Visual", "Peak1", "Peak2", "Comments", "Reference", 
    #             "CTI_Back_Left_Mean", "CTI_Back_Left_RMS", "CTI_Back_Left_Skewness", "CTI_Back_Left_Integral", 
    #             "CTI_Back_Below_Mean", "CTI_Back_Below_RMS", "CTI_Back_Below_Skewness", "CTI_Back_Below_Integral",
    #             "CTI_Front_Left_Fraction", "CTI_Front_Right_Fraction", "CTI_Front_Above_Fraction", "CTI_Front_Below_Fraction",
    #             "CTIx_Comments", "CTIy_Comments"]
    # types = ["VARCHAR(4)", "VARCHAR(4)", "FLOAT", "FLOAT", "TEXT", "VARCHAR(100)",
    #          "FLOAT", "FLOAT", "FLOAT", "FLOAT",
    #          "FLOAT", "FLOAT", "FLOAT", "FLOAT",
    #          "FLOAT", "FLOAT", "FLOAT", "FLOAT",
    #          "TEXT", "TEXT"]
    
    # image_number = 7
    # suffixes = ["Peak1", "Peak2", "Comments", "Reference", 
    #             "CTI_Back_Left_Mean", "CTI_Back_Left_RMS", "CTI_Back_Left_Skewness", "CTI_Back_Left_Integral", 
    #             "CTI_Back_Below_Mean", "CTI_Back_Below_RMS", "CTI_Back_Below_Skewness", "CTI_Back_Below_Integral",
    #             "CTI_Front_Left_Fraction", "CTI_Front_Right_Fraction", "CTI_Front_Above_Fraction", "CTI_Front_Below_Fraction",
    #             "CTIx_Comments", "CTIy_Comments"]
    # types = ["FLOAT", "FLOAT", "TEXT", "VARCHAR(100)",
    #          "FLOAT", "FLOAT", "FLOAT", "FLOAT",
    #          "FLOAT", "FLOAT", "FLOAT", "FLOAT",
    #          "FLOAT", "FLOAT", "FLOAT", "FLOAT",
    #          "TEXT", "TEXT"]
    
    # image_number = 5
    # suffixes = ["Crosstalk_01", "Crosstalk_02", "Crosstalk_03", "Crosstalk_12", "Crosstalk_13", "Crosstalk_23", "Crosstalk_Comments",]
    # types = ["FLOAT", "FLOAT", "FLOAT", "FLOAT", "FLOAT", "FLOAT", "TEXT"]
    # amps = [" "]

            



    lines = [f"ALTER TABLE {table_name}"]
    for temp in temperatures:
        for amp in amps:
            for i in range (len(suffixes)):
                suffix = suffixes[i]
                type = types[i]
                col_name = f"Image{image_number}_{temp}_{suffix}_{amp}"
                # lines.append(f"  ADD COLUMN `{col_name}` {type} DEFAULT NULL,")
                lines.append(f"  MODIFY COLUMN `{col_name}`{type},")
                
    
    return "\n".join(lines)

print(generate_sql_block())

ALTER TABLE MODULE_UNDERGROUND2
  MODIFY COLUMN `Image32_High_Res_A`FLOAT,
  MODIFY COLUMN `Image32_High_Gain_A`FLOAT,
  MODIFY COLUMN `Image32_High_Dark_Current_A`FLOAT,
  MODIFY COLUMN `Image32_High_Res_B`FLOAT,
  MODIFY COLUMN `Image32_High_Gain_B`FLOAT,
  MODIFY COLUMN `Image32_High_Dark_Current_B`FLOAT,
  MODIFY COLUMN `Image32_High_Res_C`FLOAT,
  MODIFY COLUMN `Image32_High_Gain_C`FLOAT,
  MODIFY COLUMN `Image32_High_Dark_Current_C`FLOAT,
  MODIFY COLUMN `Image32_High_Res_D`FLOAT,
  MODIFY COLUMN `Image32_High_Gain_D`FLOAT,
  MODIFY COLUMN `Image32_High_Dark_Current_D`FLOAT,
  MODIFY COLUMN `Image32_Low_Res_A`FLOAT,
  MODIFY COLUMN `Image32_Low_Gain_A`FLOAT,
  MODIFY COLUMN `Image32_Low_Dark_Current_A`FLOAT,
  MODIFY COLUMN `Image32_Low_Res_B`FLOAT,
  MODIFY COLUMN `Image32_Low_Gain_B`FLOAT,
  MODIFY COLUMN `Image32_Low_Dark_Current_B`FLOAT,
  MODIFY COLUMN `Image32_Low_Res_C`FLOAT,
  MODIFY COLUMN `Image32_Low_Gain_C`FLOAT,
  MODIFY COLUMN `Image32_Low_Dark_Current_C`FLOAT,
  MO

In [4]:
import mysql.connector

def get_existing_columns(host, user, password, database, table):
    connection = mysql.connector.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = connection.cursor()
    cursor.execute(f"SHOW COLUMNS FROM {table}")
    columns = {row[0] for row in cursor.fetchall()}
    cursor.close()
    connection.close()
    return columns

def generate_missing_columns(image_number: int, temperature: str, existing_columns: set):
    suffixes = ["Defects", "CTI_Visual", "Comments", "Reference", "Peak1", "Peak2", "Sigma", "Front"]
    sections = ["A", "B", "C", "D"]
    
    new_columns = []
    for section in sections:
        for suffix in suffixes:
            col_name = f"Image{image_number}_{temperature}_{suffix}_{section}"
            if col_name not in existing_columns:
                new_columns.append(col_name)

    # Add file field if needed
    file_col = f"Image{image_number}_{temperature}_File"
    if file_col not in existing_columns:
        new_columns.append(file_col)

    return new_columns

def add_columns_to_table(host, user, password, database, table, image_number, temperature):
    existing = get_existing_columns(host, user, password, database, table)
    new_columns = generate_missing_columns(image_number, temperature, existing)
    
    if not new_columns:
        print("-- All columns already exist. Nothing to add.")
        return
    
    sql = f"ALTER TABLE {table}\n" + ",\n".join(
        [f"  ADD COLUMN `{col}` TEXT" for col in new_columns]
    ) + ";"

    print("Executing SQL:\n", sql)

    connection = mysql.connector.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = connection.cursor()
    cursor.execute(sql)
    connection.commit()
    cursor.close()
    connection.close()
    print(f"-- Successfully added {len(new_columns)} columns.")


In [5]:
## comment it out when unused
# # --- Replace with your credentials and inputs ---
# host = "localhost"
# user = "root"
# password = "MyLife4Aiur"
# database = "die_qc"
# table = "MODULE_UNDERGROUND"

# image_number = 99
# temperature = "Low"  # or "High"

# add_columns_to_table(host, user, password, database, table, image_number, temperature)


In [ ]:
def generate_php_block(image_number: int, temperature: str, skip: str, binning: str,
                       resolution: str, exposure: str, aim: str) -> str:
    image_label = f"Image {image_number}"
    image_id = f"image{image_number}_{temperature.lower()}"
    description = f"{skip}skip, {binning}x binning, {resolution}, Active region, {exposure}s Exposure"

    return f"""
    <!-- {image_label} -  -->
    <?php echo "<b>{image_label}, {temperature} Temp - [{description}] - Aim: {aim}</b>"; ?>
    <form action="<?php echo $_SERVER['PHP_SELF']; ?>" method="post" enctype="multipart/form-data">
    <input type="hidden" name="id" value="<?php echo $id; ?>">
    <table border="1">
        <tr>
            <td align="left" style="width: 10%; white-space: nowrap;">Amplifier</td>
            <td align="left" style="width: 5%;">Defects?</td>
            <td align="left" style="width: 5%;">CTI? - Visual</td>
            <td align="left" style="width: 5%;">Energy Peak 1 [keV]</td>
            <td align="left" style="width: 5%;">Energy Peak 2 [keV]</td>
            <td align="left" style="width: 5%;">Sigma - Back Events [pixels]</td>
            <td align="left" style="width: 5%;">Front Events?</td>
            <td align="left" style="width: 25%;">Comments</td>
            <td align="left" style="width: 25%;">Reference Image</td>
        </tr>
        <?php
        $count = 0;
        $count_plus = 1;
        foreach ($ccds as $amp):
        ?>
        <tr>
            <td><?php echo "ch" . $count . " (ext" . $count_plus . ")"; ?></td>
            <td><?php generate_dropdown('{image_id}_defects_' . $amp, $yes_no_blank_array, ${{'{image_id}_defects_' . $amp}}); ?></td>
            <td><?php generate_dropdown('{image_id}_cti_visual_' . $amp, $yes_no_blank_array, ${{'{image_id}_cti_visual_' . $amp}}); ?></td>
            <td><input type="text" name="{image_id}_peak1_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_peak1_' . $amp}}; ?>" size="15"></td>
            <td><input type="text" name="{image_id}_peak2_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_peak2_' . $amp}}; ?>" size="15"></td>
            <td><input type="text" name="{image_id}_sigma_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_sigma_' . $amp}}; ?>" size="15"></td>
            <td><?php generate_dropdown('{image_id}_front_' . $amp, $yes_no_blank_array, ${{'{image_id}_front_' . $amp}}); ?></td>
            <td><input type="text" name="{image_id}_comments_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_comments_' . $amp}}; ?>" size="40"></td>
            <td><input type="text" name="{image_id}_reference_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_reference_' . $amp}}; ?>" size="50"></td>
        </tr>
        <?php $count++; $count_plus++; endforeach; ?>

        <tr>
            <td align="left" colspan="9" style="border: none; white-space: nowrap;">
                <input type="submit" value="Submit">
                <?php
		        $module_underground_id = isset($_SESSION['choosen_module_underground']) ? $_SESSION['choosen_module_underground'] : 0;
                // Absolute path on the server's file system
		        $upload_dir = '/var/www/html/QC_production/uploads/edit_module_underground/module_underground_' . $module_underground_id . '/';  // This should be the actual server file path
		        // Web URL for accessing files via the browser
                $base_url   = '/QC_production/uploads/edit_module_underground/' . 'module_underground_' . $module_underground_id; // Web URL

                ${{'{image_id}_file_name'}} = "{image_id}_file.png";
                ${{'{image_id}_file_path'}} = $upload_dir . ${{'{image_id}_file_name'}};
                ${{'{image_id}_log_name'}} = "{image_id}_log.log";
                ${{'{image_id}_log_path'}} = $upload_dir . ${{'{image_id}_log_name'}};

                if (file_exists(${{'{image_id}_file_path'}}) && !isset($_SESSION['file_url_' . $module_underground_id]['{image_id}_file'])) {{
                    $_SESSION['file_url_' . $module_underground_id]['{image_id}_file'] = $base_url . ${{'{image_id}_file_name'}};
                }}
                if (!empty(${{'{image_id}_file_name'}}) && file_exists(${{'{image_id}_file_path'}})) {{
                    $file_exists = true;
                    $file_url = $_SESSION['file_url_' . $module_underground_id]['{image_id}_file'];
                }} else {{
                    $file_exists = false;
                }}

                if (file_exists(${{'{image_id}_log_path'}}) && !isset($_SESSION['log_url_' . $module_underground_id]['{image_id}_log'])) {{
                    $_SESSION['log_url_' . $module_underground_id]['{image_id}_log'] = $base_url . ${{'{image_id}_log_name'}};
                }}
                if (!empty(${{'{image_id}_log_name'}}) && file_exists(${{'{image_id}_log_path'}})) {{
                    $log_exists = true;
                    $log_url = $_SESSION['log_url_' . $module_underground_id]['{image_id}_log'];
                }} else {{
                    $log_exists = false;
                }}
                ?>
                <?php if ($file_exists): ?>
                    <a href="<?php echo htmlspecialchars($file_url); ?>" target="_blank">
                        <img src="pixmaps/icon.png" alt="{image_label}_{temperature} File" style="height: 20px;">
                    </a>
                <?php endif; ?>
                <label for="{image_id}_file">Image File:</label>
                <input type="file" name="{image_id}_file" accept="image/png, image/jpeg, application/pdf">

                <?php if ($log_exists): ?>
                    <a href="<?php echo htmlspecialchars($log_url); ?>" target="_blank">
                        <img src="pixmaps/icon2.png" alt="{image_label}_{temperature} Log" style="height: 20px;">
                    </a>
                <?php endif; ?>
                <label for="{image_id}_log">Log File:</label>
                <input type="file" name="{image_id}_log" accept=".log,text/plain">
            </td>
        </tr>
    </table>
</form>
<br><br>
<!-- Insert the code block of next image after this -->
"""


In [6]:
print(generate_php_block(
    image_number=7,
    temperature="Low",
    skip="500",
    binning="1x10",
    resolution="640rx320c",
    exposure="500",
    aim="High Resolution Fe55 Cluster Analysis, CTI, Noise"
))

<!-- Image 7 -  -->
<?php echo "<b>Image 7, Low Temp - [500skip, 1x10x binning, 640rx320c, Active region, 500s Exposure] - Aim: High Resolution Fe55 Cluster Analysis, CTI, Noise</b>"; ?>
<form action="<?php echo $_SERVER['PHP_SELF']; ?>" method="post" enctype="multipart/form-data">
    <input type="hidden" name="id" value="<?php echo $id; ?>">
    <table border="1">
        <tr>
            <td align="left" style="width: 10%; white-space: nowrap;">Amplifier</td>
            <td align="left" style="width: 5%;">Defects?</td>
            <td align="left" style="width: 5%;">CTI? - Visual</td>
            <td align="left" style="width: 5%;">Energy Peak 1 [keV]</td>
            <td align="left" style="width: 5%;">Energy Peak 2 [keV]</td>
            <td align="left" style="width: 5%;">Sigma - Back Events [pixels]</td>
            <td align="left" style="width: 5%;">Front Events?</td>
            <td align="left" style="width: 25%;">Comments</td>
            <td align="left" style="width: 25%

In [1]:
import mysql.connector
from time import sleep

def chunk_list(lst, n):
    """Yield successive n-sized chunks from list."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

# --- CONFIGURATION ---
db_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'MyLife4Aiur',
    'database': 'die_qc'
}

table_name = 'MODULE_UNDERGROUND'
preserve_column = 'Name'

# --- CONNECT TO DATABASE ---
conn = mysql.connector.connect(**db_config)
cursor = conn.cursor()

# --- GET ALL COLUMNS ---
cursor.execute(f"SHOW COLUMNS FROM {table_name}")
columns_info = cursor.fetchall()

# --- FILTER NULLABLE COLUMNS (excluding 'Name') ---
columns_to_null = [
    col[0] for col in columns_info
    if col[0] != preserve_column and col[2].upper() == 'YES'
]

print(f"Found {len(columns_to_null)} nullable columns to clear.")

# --- UPDATE IN CHUNKS OF 10 ---
for i, chunk in enumerate(chunk_list(columns_to_null, 1), start=1):
    set_clause = ",\n  ".join([f"`{col}` = NULL" for col in chunk])
    sql = f"UPDATE {table_name} SET\n  {set_clause};"
    
    print(f"\n⏳ Executing chunk {i}: {len(chunk)} columns")
    try:
        cursor.execute(sql)
        conn.commit()
        print(f"✔ Chunk {i} succeeded.")
    except mysql.connector.Error as e:
        print(f"❌ Chunk {i} failed: {e}")
        break  # Stop on failure
    sleep(1)  # optional: delay to reduce locking issues

cursor.close()
conn.close()


Found 592 nullable columns to clear.

⏳ Executing chunk 1: 1 columns
✔ Chunk 1 succeeded.

⏳ Executing chunk 2: 1 columns
✔ Chunk 2 succeeded.

⏳ Executing chunk 3: 1 columns
✔ Chunk 3 succeeded.

⏳ Executing chunk 4: 1 columns
✔ Chunk 4 succeeded.

⏳ Executing chunk 5: 1 columns
✔ Chunk 5 succeeded.

⏳ Executing chunk 6: 1 columns
✔ Chunk 6 succeeded.

⏳ Executing chunk 7: 1 columns
✔ Chunk 7 succeeded.

⏳ Executing chunk 8: 1 columns
✔ Chunk 8 succeeded.

⏳ Executing chunk 9: 1 columns
✔ Chunk 9 succeeded.

⏳ Executing chunk 10: 1 columns
✔ Chunk 10 succeeded.

⏳ Executing chunk 11: 1 columns
✔ Chunk 11 succeeded.

⏳ Executing chunk 12: 1 columns
✔ Chunk 12 succeeded.

⏳ Executing chunk 13: 1 columns
✔ Chunk 13 succeeded.

⏳ Executing chunk 14: 1 columns
✔ Chunk 14 succeeded.

⏳ Executing chunk 15: 1 columns
✔ Chunk 15 succeeded.

⏳ Executing chunk 16: 1 columns
✔ Chunk 16 succeeded.

⏳ Executing chunk 17: 1 columns
✔ Chunk 17 succeeded.

⏳ Executing chunk 18: 1 columns
✔ Chunk 18 su